### Mini project using NumPy for solar irradiance data analysis

##### Purpose: This notebook demonstrates NumPy data cleaning and analysis on solar irradiance data for multiple sites.
##### Tools: Built using NumPy only, no Pandas, for fundamental learning.

In [1]:
import numpy as np 

#### Functions
- load data
- clean data
- calculate statistics
- find low irradiance
- find high irradiance

In [2]:
# Loads CSV irradiance data 
def read_file(file_path): 
    datetime = np.genfromtxt(
        file_path,
        delimiter=",",
        skip_header=1,
        usecols=(0),
        dtype=str
    )
    
    irradiance = np.genfromtxt(
        file_path,
        delimiter=",",
        skip_header=1,
        usecols=(1, 2, 3) 
    )
    
    return datetime.reshape(-1, 1), irradiance

# Sets values below the threshold to NaN.
def clean_low_irradiance(irradiance_arr, threshold): 
    mask = irradiance_arr <= threshold 
    irradiance_arr[mask] = np.nan
    
    return irradiance_arr 

# Calculates mean, max, and valid count ignoring NaNs.
def compute_statistics(irradiance_arr):
    min_values = np.nanmin(irradiance_arr, axis=0)
    max_values = np.nanmax(irradiance_arr, axis=0)
    mean_values = np.nanmean(irradiance_arr, axis=0)
    count_values = np.sum(~np.isnan(irradiance_arr), axis=0)
    
    results = dict() 
    for i in range(np.shape(irradiance_arr)[1]): 
        results["site_" + str(i+1)] = { 
            "min": min_values[i], 
            "max": max_values[i], 
            "mean": mean_values[i],
            "count": count_values[i]
        }

    return results

# Identifies daytime where all sites have irradiance below the threshold.
def find_low_irradiance(datetime_arr, irradiance_arr, threshold):
    # Extract times from datetime
    times = np.array([int(time.split()[1].split(":")[0]) for time in datetime_arr.flatten()])
    times_mask = (times >= 10) & (times <= 14)
    
    irradiance_mask = np.all((irradiance_arr<threshold), axis=1)
    combined_mask = times_mask & irradiance_mask
    
    datetime_arr = datetime_arr[combined_mask].flatten().tolist()
    irradiance_arr = irradiance_arr[combined_mask].tolist()
    
    return {
        "low_irradiance": {
            datetime: irradiance for datetime, irradiance in zip(datetime_arr, irradiance_arr)
        }
    }
    
# Identifies daytime where all sites have irradiance above the threshold.
def find_high_irradiance(datetime_arr, irradiance_arr, threshold):
    # Extract times from datetime
    times = np.array([int(time.split()[1].split(":")[0]) for time in datetime_arr.flatten()])
    times_mask = (times >= 10) & (times <= 14)
    
    irradiance_mask = np.all((irradiance_arr>threshold), axis=1)
    combined_mask = times_mask & irradiance_mask
    
    datetime_arr = datetime_arr[combined_mask].flatten().tolist()
    irradiance_arr = irradiance_arr[combined_mask].tolist()
    
    return {
        "high_irradiance": {
            datetime: irradiance for datetime, irradiance in zip(datetime_arr, irradiance_arr)
        }
    }

##### Load data

In [3]:
datetime_data, irradiance_data = read_file("irradiance_data.csv")

##### Clean low irradiance values

In [4]:
irradiance_data = clean_low_irradiance(irradiance_data, 0)

##### Calculate statistics

In [5]:
result_statistics = compute_statistics(irradiance_data)
print(f"site_statistics: {result_statistics}")

site_statistics: {'site_1': {'min': 1.507, 'max': 1124.5, 'mean': 405.1513257115981, 'count': 4251}, 'site_2': {'min': 3.5962, 'max': 1083.1, 'mean': 407.9542641678463, 'count': 4242}, 'site_3': {'min': 7.0158, 'max': 1078.0, 'mean': 405.4634028261898, 'count': 4246}}


##### Identify low irradiance

In [6]:
low_irradiance_threshold = 70 # in W/m², adjust threshold for low irradiance filtering
result_low_irradiance = find_low_irradiance(datetime_data, irradiance_data, low_irradiance_threshold)
print(f"low irradiance time and irradiance: {result_low_irradiance}")

low irradiance time and irradiance: {'low_irradiance': {'11/8/1990 14:00': [68.791, 64.24, 64.221]}}


##### Identify high irradiance

In [7]:
high_irradiance_threshold = 988 # in W/m², adjust threshold for high irradiance filtering
result_high_irradiance = find_high_irradiance(datetime_data, irradiance_data, high_irradiance_threshold)
print(f"high irradiance time and irradiance: {result_high_irradiance}")

high irradiance time and irradiance: {'high_irradiance': {'16/06/90 12:00': [988.57, 1008.8, 1015.0]}}
